# Retail Analytics: End-to-End Pipeline
Data cleaning, RFM/churn analysis, and MySQL loading for a grocery retail 
analytics project using the Dunnhumby "Complete Journey" dataset (2,500 
households, 2 years of transactions).

In [1]:
# Install kaggle 
!pip install kaggle

## 1. Data Ingestion
Downloading the raw dataset via the Kaggle API.

In [2]:
!kaggle datasets download -d frtgnn/dunnhumby-the-complete-journey -p ../data/raw --unzip

Dataset URL: https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey
License(s): DbCL-1.0
100%|███████████████████████████████████████| 124M/124M [00:05<00:00, 22.3MB/s]



Confirming all 8 raw CSV files downloaded successfully.

In [3]:
import os
raw_files = os.listdir('../data/raw')
print(raw_files)

['hh_demographic.csv', 'causal_data.csv', 'coupon.csv', 'campaign_table.csv', 'coupon_redempt.csv', 'product.csv', 'campaign_desc.csv', 'transaction_data.csv']


## 2. Initial Data Inspection
Previewing shape, columns, and sample rows for each raw file before any cleaning.

In [4]:
import pandas as pd
import os

for f in os.listdir('../data/raw'):
    df = pd.read_csv(f'../data/raw/{f}')
    print(f"=== {f} ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(2))
    print()

=== hh_demographic.csv ===
Shape: (801, 8)
Columns: ['AGE_DESC', 'MARITAL_STATUS_CODE', 'INCOME_DESC', 'HOMEOWNER_DESC', 'HH_COMP_DESC', 'HOUSEHOLD_SIZE_DESC', 'KID_CATEGORY_DESC', 'household_key']
  AGE_DESC MARITAL_STATUS_CODE INCOME_DESC HOMEOWNER_DESC      HH_COMP_DESC  \
0      65+                   A      35-49K      Homeowner  2 Adults No Kids   
1    45-54                   A      50-74K      Homeowner  2 Adults No Kids   

  HOUSEHOLD_SIZE_DESC KID_CATEGORY_DESC  household_key  
0                   2      None/Unknown              1  
1                   2      None/Unknown              7  

=== causal_data.csv ===
Shape: (36786524, 5)
Columns: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']
   PRODUCT_ID  STORE_ID  WEEK_NO display mailer
0       26190       286       70       0      A
1       26190       288       70       0      A

=== coupon.csv ===
Shape: (124548, 3)
Columns: ['COUPON_UPC', 'PRODUCT_ID', 'CAMPAIGN']
    COUPON_UPC  PRODUCT_ID  CAMPAIGN
0  100000

## 3. Loading & Standardizing
Loading all 8 files into a dictionary of DataFrames and standardizing column 
names to lowercase for consistency across files (Dunnhumby uses inconsistent 
casing between tables).

In [5]:
import pandas as pd
import os

# Load all raw files
files = {
    'households': 'hh_demographic.csv',
    'products': 'product.csv',
    'transactions': 'transaction_data.csv',
    'campaigns': 'campaign_desc.csv',
    'campaign_household': 'campaign_table.csv',
    'coupons': 'coupon.csv',
    'coupon_redemptions': 'coupon_redempt.csv',
    'causal': 'causal_data.csv'
}

dfs = {}
for name, filename in files.items():
    df = pd.read_csv(f'../data/raw/{filename}')
    df.columns = df.columns.str.lower()  # standardize to lowercase
    dfs[name] = df
    print(f"{name}: {df.shape}")

households: (801, 8)
products: (92353, 7)
transactions: (2595732, 12)
campaigns: (30, 4)
campaign_household: (7208, 3)
coupons: (124548, 3)
coupon_redemptions: (2318, 4)
causal: (36786524, 5)


## 4. Data Quality Checks
Checking data types and null values across all 8 tables. Note: this dataset 
uses placeholder strings (e.g. "None/Unknown") rather than true NaN for 
missing values, so a "zero nulls" result doesn't mean zero missing data.

In [6]:
for name, df in dfs.items():
    print(f"=== {name} ===")
    print(df.dtypes)
    print("Nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])
    print()

=== households ===
age_desc                 str
marital_status_code      str
income_desc              str
homeowner_desc           str
hh_comp_desc             str
household_size_desc      str
kid_category_desc        str
household_key          int64
dtype: object
Nulls:
 Series([], dtype: int64)

=== products ===
product_id              int64
manufacturer            int64
department                str
brand                     str
commodity_desc            str
sub_commodity_desc        str
curr_size_of_product      str
dtype: object
Nulls:
 Series([], dtype: int64)

=== transactions ===
household_key          int64
basket_id              int64
day                    int64
product_id             int64
quantity               int64
sales_value          float64
store_id               int64
retail_disc          float64
trans_time             int64
week_no                int64
coupon_disc          float64
coupon_match_disc    float64
dtype: object
Nulls:
 Series([], dtype: int64)

=== campa

## 5. Cleaning Individual Tables

### Households
Renaming columns for clarity and removing duplicate household records.
Note: demographic data is only available for 801 of the 2,500 households.

In [7]:
households = dfs['households'].copy()
households = households.rename(columns={
    'household_key': 'household_key',
    'age_desc': 'age_desc',
    'marital_status_code': 'marital_status',
    'income_desc': 'income_desc',
    'homeowner_desc': 'homeowner_desc',
    'hh_comp_desc': 'hh_comp_desc',
    'household_size_desc': 'household_size',
    'kid_category_desc': 'kid_category'
})
households = households.drop_duplicates(subset='household_key')
print(households.shape)

(801, 8)


### Products
Stripping whitespace inconsistencies in category/description fields and 
removing duplicate product records.

In [8]:
products = dfs['products'].copy()
products['department'] = products['department'].str.strip()
products['commodity_desc'] = products['commodity_desc'].str.strip()
products['sub_commodity_desc'] = products['sub_commodity_desc'].str.strip()
products = products.drop_duplicates(subset='product_id')
print(products.shape)
print(products.isnull().sum())

(92353, 7)
product_id              0
manufacturer            0
department              0
brand                   0
commodity_desc          0
sub_commodity_desc      0
curr_size_of_product    0
dtype: int64


### Transactions
Checking for data quality issues — negative sales values, zero-quantity rows, 
and the overall date range.

In [9]:
transactions = dfs['transactions'].copy()

# Check for negative/zero sales values (returns or data errors)
print("Negative sales_value rows:", (transactions['sales_value'] < 0).sum())
print("Zero quantity rows:", (transactions['quantity'] == 0).sum())

# Check date range
print("Day range:", transactions['day'].min(), "-", transactions['day'].max())

Negative sales_value rows: 0
Zero quantity rows: 14466
Day range: 1 - 711


Investigating the 14,466 zero-quantity rows in detail — these turned out to 
be coupon-only adjustment line items (no actual product purchased), not 
genuine transactions or data errors.

In [10]:
zero_qty = transactions[transactions['quantity'] == 0]
print(zero_qty[['quantity', 'sales_value', 'retail_disc', 'coupon_disc']].describe())
print()
print(zero_qty.head(5))

       quantity   sales_value   retail_disc   coupon_disc
count   14466.0  14466.000000  14466.000000  14466.000000
mean        0.0      0.001333      0.000169     -0.713987
std         0.0      0.059967      0.022250      1.682890
min         0.0      0.000000     -0.860000    -37.930000
25%         0.0      0.000000      0.000000     -1.000000
50%         0.0      0.000000      0.000000      0.000000
75%         0.0      0.000000      0.000000      0.000000
max         0.0      5.820000      2.090000      0.000000

     household_key    basket_id  day  product_id  quantity  sales_value  \
97             744  26985165432    1     5978648         0          0.0   
128           1287  26985336468    1     5978648         0          0.0   
249           2305  26996870743    2     5978656         0          0.0   
293            271  26997082949    2     5978656         0          0.0   
694            315  27008952267    3      957951         0          0.0   

     store_id  retail_disc

Dropping the zero-quantity rows and adding a `net_sales_value` column that 
reflects actual revenue after retail and coupon discounts are applied.

In [11]:
transactions_clean = transactions[transactions['quantity'] > 0].copy()
print("Before:", transactions.shape)
print("After:", transactions_clean.shape)

Before: (2595732, 12)
After: (2581266, 12)


In [12]:
transactions_clean['net_sales_value'] = (
    transactions_clean['sales_value'] 
    + transactions_clean['retail_disc'] 
    + transactions_clean['coupon_disc']
)

### Promotion Data (Causal)
Examining the display/mailer promo codes before simplifying them — `0` means 
no promotion, other codes represent different promo placement types.

In [13]:
causal = dfs['causal'].copy()
print(causal['display'].value_counts())
print(causal['mailer'].value_counts())

display
0    21038745
9     2699467
5     2575289
7     2362118
3     2073738
6     1816021
2     1812840
1     1102141
A      713180
4      592985
Name: count, dtype: int64
mailer
A    17106789
0    11534183
D     4467453
H     1560395
F     1077549
J      306924
L      301327
C      291059
X      120823
Z       19453
P         569
Name: count, dtype: int64


Simplifying the granular promo codes into simple boolean flags: was a 
product displayed and/or featured in a mailer, regardless of the specific 
placement type.

In [14]:
causal_clean = causal.copy()

causal_clean['had_display'] = causal_clean['display'] != '0'
causal_clean['had_mailer'] = causal_clean['mailer'] != '0'

causal_agg = causal_clean[['product_id', 'store_id', 'week_no', 'had_display', 'had_mailer']].copy()

print(causal_agg.shape)
print(causal_agg[['had_display','had_mailer']].mean())  # gives % of rows with any promo

(36786524, 5)
had_display    0.428086
had_mailer     0.686456
dtype: float64


Filtering to only products that appear in the actual transactions data, 
then aggregating from 36.7M product-store-week rows down to ~3.9M 
product-store level summaries (% of tracked weeks with display/mailer 
activity). This makes the promo data usable for pricing analysis without 
needing week-by-week granularity.

In [15]:
transacted_products = transactions_clean['product_id'].unique()
causal_final = causal_agg[causal_agg['product_id'].isin(transacted_products)].copy()
print("Before filter:", causal_agg.shape)
print("After filter:", causal_final.shape)

Before filter: (36786524, 5)
After filter: (36771881, 5)


In [16]:
causal_summary = causal_final.groupby(['product_id', 'store_id']).agg(
    weeks_tracked=('week_no', 'count'),
    weeks_displayed=('had_display', 'sum'),
    weeks_mailer=('had_mailer', 'sum')
).reset_index()

causal_summary['pct_weeks_displayed'] = causal_summary['weeks_displayed'] / causal_summary['weeks_tracked']
causal_summary['pct_weeks_mailer'] = causal_summary['weeks_mailer'] / causal_summary['weeks_tracked']

print(causal_summary.shape)
print(causal_summary.head())

(3930718, 7)
   product_id  store_id  weeks_tracked  weeks_displayed  weeks_mailer  \
0       26190       286              1                0             1   
1       26190       288              1                0             1   
2       26190       289              1                0             1   
3       26190       292              1                0             1   
4       26190       293              1                0             1   

   pct_weeks_displayed  pct_weeks_mailer  
0                  0.0               1.0  
1                  0.0               1.0  
2                  0.0               1.0  
3                  0.0               1.0  
4                  0.0               1.0  


## 6. Loading Cleaned Data into MySQL
Connecting to the local MySQL database and confirming the schema's 8 tables 
exist before loading data.

In [ ]:
from sqlalchemy import create_engine, text

username = 'retail_user'
password = 'YOUR_PASSWORD_HERE'  # replace with your own local MySQL password before running
host = 'localhost'
port = '3306'
database = 'retail_analytics'

engine = create_engine(f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}')

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES;"))
    for row in result:
        print(row)

('campaign_household',)
('campaigns',)
('coupon_redemptions',)
('coupons',)
('customer_rfm_churn',)
('households',)
('products',)
('promo_effectiveness',)
('promo_summary',)
('transactions',)


Loading each cleaned table into its corresponding MySQL table.

In [18]:
households.to_sql('households', engine, if_exists='append', index=False)
print("households loaded")

products.to_sql('products', engine, if_exists='append', index=False)
print("products loaded")

dfs['campaigns'].to_sql('campaigns', engine, if_exists='append', index=False)
print("campaigns loaded")

dfs['campaign_household'].to_sql('campaign_household', engine, if_exists='append', index=False)
print("campaign_household loaded")

dfs['coupons'].to_sql('coupons', engine, if_exists='append', index=False)
print("coupons loaded")

dfs['coupon_redemptions'].to_sql('coupon_redemptions', engine, if_exists='append', index=False)
print("coupon_redemptions loaded")

causal_summary.to_sql('promo_summary', engine, if_exists='append', index=False)
print("promo_summary loaded")

households loaded
products loaded
campaigns loaded
campaign_household loaded
coupons loaded
coupon_redemptions loaded
promo_summary loaded


Loading transactions last, in chunks, given its size (~2.58M rows).

In [19]:
transactions_clean.to_sql(
    'transactions',
    engine,
    if_exists='append',
    index=False,
    chunksize=50000,
    method='multi'
)
print("transactions loaded")

transactions loaded


## 7. RFM Analysis & Customer Segmentation
Using the last transaction day in the dataset as the snapshot/reference 
point for calculating recency.

In [20]:
snapshot_day = transactions_clean['day'].max()
print("Snapshot day:", snapshot_day)

Snapshot day: 711


Calculating Recency (days since last purchase), Frequency (distinct 
baskets), and Monetary (total net spend) for each of the 2,500 households.

In [21]:
rfm = transactions_clean.groupby('household_key').agg(
    recency=('day', lambda x: snapshot_day - x.max()),
    frequency=('basket_id', 'nunique'),
    monetary=('net_sales_value', 'sum')
).reset_index()

print(rfm.shape)
print(rfm.head())
print(rfm.describe())

(2500, 4)
   household_key  recency  frequency  monetary
0              1        5         85   3559.81
1              2       43         45   1610.35
2              3        8         47   1916.86
3              4       84         30   1081.96
4              5        8         40    660.73
       household_key      recency    frequency      monetary
count     2500.00000  2500.000000  2500.000000   2500.000000
mean      1250.50000    25.576000   110.355600   2650.729400
std        721.83216    62.791673   115.433111   2857.319402
min          1.00000     0.000000     1.000000      7.920000
25%        625.75000     1.000000    38.000000    759.882500
50%       1250.50000     6.000000    78.000000   1735.510000
75%       1875.25000    20.000000   142.000000   3604.237500
max       2500.00000   657.000000  1298.000000  36152.440000


Defining churn as no purchase in the last 90+ days — a reasonable threshold 
given the 75th percentile of recency across all households is 20 days.

In [22]:
rfm['churned'] = rfm['recency'] > 90
print(rfm['churned'].value_counts())
print(rfm['churned'].mean())  # % churned

churned
False    2329
True      171
Name: count, dtype: int64
0.0684


Scoring each household 1–5 on Recency, Frequency, and Monetary using 
quintile-based binning, then summing into a combined RFM score (3–15).

In [23]:
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1]).astype(int)  # lower recency = higher score
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5]).astype(int)

rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

print(rfm[['r_score','f_score','m_score','rfm_score']].describe())

           r_score      f_score      m_score   rfm_score
count  2500.000000  2500.000000  2500.000000  2500.00000
mean      3.102000     3.000000     3.000000     9.10200
std       1.468489     1.414496     1.414496     3.67132
min       1.000000     1.000000     1.000000     3.00000
25%       2.000000     2.000000     2.000000     6.00000
50%       3.000000     3.000000     3.000000     9.00000
75%       5.000000     4.000000     4.000000    12.00000
max       5.000000     5.000000     5.000000    15.00000


Segmenting households into 5 groups (Champions, Loyal, Potential, At Risk, 
Churned) based on RFM score, with churn status overriding other segments - 
so a customer's historical value doesn't hide the fact they've gone quiet.

In [24]:
def segment_customer(row):
    if row['churned']:
        return 'Churned'
    elif row['rfm_score'] >= 13:
        return 'Champions'
    elif row['rfm_score'] >= 10:
        return 'Loyal'
    elif row['rfm_score'] >= 7:
        return 'Potential'
    else:
        return 'At Risk'

rfm['segment'] = rfm.apply(segment_customer, axis=1)
print(rfm['segment'].value_counts())

segment
Loyal        646
Potential    583
Champions    561
At Risk      539
Churned      171
Name: count, dtype: int64


Loading the RFM/segmentation results into MySQL as a new table for use in 
the Power BI dashboard.

In [25]:
rfm.to_sql('customer_rfm_churn', engine, if_exists='replace', index=False)
print("customer_rfm_churn loaded")

customer_rfm_churn loaded


In [26]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM customer_rfm_churn;"))
    print(result.fetchone())

(2500,)


## 8. Data Quality Investigation: Quantity Outliers
Investigating an anomaly found later while building the Power BI "units 
sold" metric — some transactions have extreme quantity values (up to 89,638) 
that skew unit-count totals.

In [27]:
print(transactions_clean['quantity'].describe())
print(transactions_clean['quantity'].sort_values(ascending=False).head(20))

count    2.581266e+06
mean     1.009914e+02
std      1.156639e+03
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      8.963800e+04
Name: quantity, dtype: float64
1750942    89638
468356     85055
481876     61335
166536     51912
1340882    48073
2472791    45475
1560340    41833
376686     41686
156049     41485
87625      39365
752631     38348
238435     37391
1736051    36140
2013726    35999
2302438    35836
1475586    35436
360965     35384
60845      35106
2124167    35077
137453     34920
Name: quantity, dtype: int64


Quantifying how many rows are affected — under 1% of transactions — before 
deciding to exclude them from unit-count metrics (revenue stays unaffected).

In [28]:
print("Rows with quantity > 100:", (transactions_clean['quantity'] > 100).sum())
print("% of total rows:", (transactions_clean['quantity'] > 100).mean() * 100)

Rows with quantity > 100: 23136
% of total rows: 0.8963043715758081


## 9. Data Quality Investigation: Misc/Fuel Category
Investigating the "COUPON/MISC ITEMS" category, which unexpectedly ranked 
#1 by revenue. Found to be dominated by gasoline sales (84% of its rows), 
not genuine grocery products — excluded from product-level analysis as a 
result.

In [29]:
misc = transactions_clean.merge(products, on='product_id')
misc_check = misc[misc['commodity_desc'] == 'COUPON/MISC ITEMS']
print(misc_check[['quantity','sales_value','net_sales_value','retail_disc','coupon_disc']].describe())
print(misc_check['sub_commodity_desc'].value_counts().head(10))

           quantity   sales_value  net_sales_value   retail_disc  coupon_disc
count  27710.000000  27710.000000     27710.000000  27710.000000      27710.0
mean    9282.498629     23.091969        22.360758     -0.731211          0.0
std     6270.298922     15.708199        15.282860      0.806283          0.0
min        1.000000      0.000000        -0.010000    -33.020000          0.0
25%     4549.000000     10.990000        10.490000     -1.270000          0.0
50%     9538.000000     22.000000        21.030000     -0.670000          0.0
75%    13605.750000     31.690000        30.477500     -0.010000          0.0
max    89638.000000    840.000000       840.000000      0.000000          0.0
sub_commodity_desc
GASOLINE-REG UNLEADED             23174
MISC SALES TRANS                   1150
MEAT SUPPLIES                       946
PRODUCE DEPT KEY RING               677
MISCELLANEOUS H & B AIDS            651
ELECTRONIC GIFT CARDS ACTIVATI      436
CAN DOG FOOD RATION (TRIX/VETS      183